# P4 — C-Lite Exhaustive Finite Leader Solve

P4가 C-lite다. 유한 sensor 후보를 하나도 건너뛰지 않고 B4 follower로 모두 풀어 finite Stackelberg optimum을 선택한다.

In [1]:
from pathlib import Path
import json
from html import escape
import subprocess
import sys
from IPython.display import display, Markdown, Image, HTML, FileLink

def locate_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "p1b_4D").is_dir() and (candidate / "p1b_roadmap_0729.md").exists():
            return candidate
    raise RuntimeError("glider_hybrid_control repository root를 찾지 못했습니다.")

ROOT = locate_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"
STAGE = "P4"
STATUS = "PENDING"

def run_module(module, *arguments, timeout=None):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    completed = subprocess.run(
        command, cwd=ROOT, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"{module} 실행 실패: exit={completed.returncode}")
    return completed.stdout

def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"결과 파일이 없습니다: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def show_png(path, width=1050):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"> ⚠️ 그림이 없습니다: `{path}`"))

def display_table(rows, columns=None):
    if isinstance(rows, dict):
        rows = [rows]
    rows = list(rows)
    if columns is None:
        columns = []
        for row in rows:
            for key in row:
                if key not in columns:
                    columns.append(key)
    if not rows:
        display(Markdown("_(표시할 행이 없습니다.)_"))
        return
    def cell(value):
        if isinstance(value, float):
            value = f"{value:.8g}"
        elif isinstance(value, (dict, list, tuple)):
            value = json.dumps(value, ensure_ascii=False)
        return escape(str(value))
    header = "".join(f"<th>{cell(name)}</th>" for name in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{cell(row.get(name, ''))}</td>" for name in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<div style='overflow-x:auto'><table>"
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    ))

display(Markdown(f"**{STAGE} 상태:** `{STATUS}`  \nRepository: `{ROOT}`"))

**P4 상태:** `PENDING`  
Repository: `C:\Users\jeffe\Desktop\git\glider_hybrid_control`

## 현재 상태

이 단계는 아직 구현 완료되지 않았다. 이 노트북은 완료된 척하지 않고,
구현 목표와 향후 실행 진입점을 한 파일에 고정한다.

**완료 조건**


- finite sensor set/bounds/spacing/tie rule 사전 고정
- 모든 leader 후보 exhaustive evaluation
- 동일한 B4 follower와 evaluator 사용
- defender objective curve, argmax, leader-grid sensitivity 출력

In [2]:
RERUN = False
EXPECTED_MODULE = ROOT / "p1b_4D" / "experiment_p4_c_lite_exhaustive_leader.py"
RESULT_PATH = ROOT / "results/p4/p4_c_lite_exhaustive_leader.json"
status = {
    "stage": "P4",
    "implementation module exists": EXPECTED_MODULE.exists(),
    "result exists": RESULT_PATH.exists(),
    "rerun requested": RERUN,
}
display_table(status)

if RERUN:
    if not EXPECTED_MODULE.exists():
        raise RuntimeError(
            f"P4 구현이 아직 없습니다: {EXPECTED_MODULE.name}"
        )
    run_module("p1b_4D.experiment_p4_c_lite_exhaustive_leader")

if RESULT_PATH.exists():
    payload = load_json(RESULT_PATH)
    display(Markdown("✅ 저장된 결과를 발견했습니다."))
    display(Markdown("```json\n" + json.dumps(payload, ensure_ascii=False, indent=2)[:12000] + "\n```"))
else:
    display(Markdown(
        "> ⏳ **PENDING:** 구현·실험이 완료되면 이 셀이 결과 표와 그림을 바로 표시합니다."
    ))

stage,implementation module exists,result exists,rerun requested
P4,False,False,False


> ⏳ **PENDING:** 구현·실험이 완료되면 이 셀이 결과 표와 그림을 바로 표시합니다.

In [3]:
manifest_path = RESULTS / "direction_b" / "b4_production_lattice_freeze.json"
manifest = load_json(manifest_path)
display(Markdown(
    f"P4가 사용할 frozen follower: `{manifest['production_configuration_id']}`"
))
display_table([
    {"setting": key, "value": value}
    for key, value in manifest["selected_settings"].items()
])

P4가 사용할 frozen follower: `direction_b_l2_enriched_v9_q9_e1025`

setting,value
action_family,enriched
endpoint_snapping,False
evaluator_sample_count,1025
level,2
planning_quadrature_count,9
speed_count,9
speed_family,V9
transition_model,successor_grid_physical_edge


## 직관적 목적

P4가 C-lite다. 유한 sensor 후보를 하나도 건너뛰지 않고 B4 follower로 모두 풀어 finite Stackelberg optimum을 선택한다.